# Day 02b: Tokenization from Scratch(Day 02b:从零构建分词器)

**Time:** ~20 minutes  
**Prerequisites:** Day 02a (What's Inside a Model)  
**Goal:** Build a BPE tokenizer by hand — understand how text becomes numbers.

> **时长:**约 20 分钟  
> **前置要求:**Day 02a(模型里面有什么)  
> **目标:**亲手实现一个 BPE 分词器——理解文本是如何变成数字的。

---

In 02a we saw that `wte.weight` has shape `[50257, 768]` — that's 50,257 rows, one per token in the vocabulary. But what is a "token"? How does `"The cat sat"` become `[464, 9246, 3332]`?

> 在 02a 中我们看到 `wte.weight` 的形状是 `[50257, 768]`——共 50,257 行,词表里每个词元对应一行。但"词元(token)"到底是什么?`"The cat sat"` 是怎么变成 `[464, 9246, 3332]` 的?

Tokenization is the **front door** to the model. No neural network involved — it's just a lookup table and a set of merge rules.

> 分词(tokenization)是模型的**大门**。这里没有任何神经网络参与——它只是一张查找表加一组合并规则。

## Why Not Just Use Characters?(为什么不能直接按字符切?)

There are three ways to split text into tokens. Each has tradeoffs — like choosing between microservices, monoliths, and serverless:

> 切分文本有三种方式,各有取舍——就像在微服务、单体和 serverless 之间做选择:

| Strategy | Tokens for "unhappiness" | Vocab Size | Problem |
|----------|--------------------------|------------|----------|
| **Characters** | `u, n, h, a, p, p, i, n, e, s, s` (11 tokens) | ~256 | Sequences are very long. Model wastes capacity learning spelling. |
| **Whole words** | `unhappiness` (1 token) | 500K+ | Vocab explodes. Can't handle new/rare words. OOV errors. |
| **Subwords (BPE)** | `un, happiness` (2 tokens) | 50K | Sweet spot. Common words = 1 token. Rare words = split into known pieces. |

> (表中对比:按字符 = "unhappiness" 被切成 11 个词元,词表仅约 256,但序列非常长,模型浪费容量去学拼写;按整词 = 1 个词元,但词表爆炸到 50 万以上,无法处理新词/生僻词,出现 OOV(词表外)错误;子词 BPE = 切成 "un" + "happiness" 2 个词元,词表约 5 万,是最佳平衡点——常见词占 1 个词元,生僻词拆成已知片段)

```
Characters  = every request gets its own pod (lots of overhead)
Whole words = one giant monolith (can't handle edge cases)
BPE         = right-sized microservices (efficient + flexible)
```

> (类比:按字符 = 每个请求独占一个 pod(开销巨大);按整词 = 一个巨型单体(处理不了边界情况);BPE = 大小合适的微服务(高效又灵活))

GPT-2 uses **Byte Pair Encoding (BPE)** with a vocabulary of 50,257 tokens.

> GPT-2 使用**字节对编码(Byte Pair Encoding,BPE)**,词表大小为 50,257 个词元。

## Step 1: Load GPT-2's Vocabulary(第一步:加载 GPT-2 的词表)

GPT-2's tokenizer needs two files from the model directory:

> GPT-2 的分词器需要模型目录下的两个文件:

- **`vocab.json`** — maps token strings to integer IDs (the lookup table)
- **`merges.txt`** — the BPE merge rules (how to combine bytes into tokens)

> - **`vocab.json`**——词元字符串到整数 ID 的映射(查找表)
> - **`merges.txt`**——BPE 合并规则(如何把字节组合成词元)

We downloaded both in notebook 01.

> 这两个文件我们在 notebook 01 中已经下载过了。

In [3]:
import json
import os

MODEL_DIR = "gpt2_weights"

with open(os.path.join(MODEL_DIR, "vocab.json")) as f:
    vocab = json.load(f)  # token string -> integer ID(词元字符串 -> 整数 ID)

with open(os.path.join(MODEL_DIR, "merges.txt")) as f:
    merges_raw = f.read().strip().split("\n")[1:]  # skip header line(跳过首行文件头)

# Reverse lookup: ID -> token string(反向查找表:ID -> 词元字符串)
id_to_token = {v: k for k, v in vocab.items()}

print(f"Vocabulary size: {len(vocab):,} tokens")
print(f"Merge rules:     {len(merges_raw):,}")
print()
print("Sample tokens (ID -> string):")
for token_id in [0, 1, 100, 464, 9246, 3332, 50256]:
    print(f"  {token_id:>6d}  ->  {repr(id_to_token[token_id])}")

Vocabulary size: 50,257 tokens
Merge rules:     50,000

Sample tokens (ID -> string):
       0  ->  '!'
       1  ->  '"'
     100  ->  '§'
     464  ->  'The'
    9246  ->  'cat'
    3332  ->  'Ġsat'
   50256  ->  '<|endoftext|>'


## What's With the Weird Characters?(那些奇怪的字符是怎么回事?)

You'll see tokens like `'Ġcat'` instead of `' cat'`. GPT-2 uses a byte-level encoding where:
- `Ġ` represents a space (byte 0x20)
- Other special chars map bytes that aren't printable ASCII

> 你会看到 `'Ġcat'` 这样的词元,而不是 `' cat'`。GPT-2 使用字节级编码:
> - `Ġ` 代表一个空格(字节 0x20)
> - 其他特殊字符对应不可打印 ASCII 的那些字节

This is GPT-2's way of encoding **every possible byte** as a printable Unicode character. It means the tokenizer can handle any input — including binary data — without special "unknown token" handling.

> 这是 GPT-2 把**每一个可能的字节**都编码成可打印 Unicode 字符的办法。这样分词器就能处理任何输入——包括二进制数据——而无需特殊的"未知词元(unknown token)"处理。

Think of it as base64 encoding for the tokenizer's internal representation.

> 可以把它理解为分词器内部表示的 base64 编码。

In [4]:
# GPT-2's byte-to-unicode mapping(GPT-2 的“字节 -> unicode 字符”映射)
def bytes_to_unicode():
    """Build GPT-2's byte <-> unicode character mapping.(构建 GPT-2 的“字节 <-> unicode 字符”映射表)
    
    Maps each byte (0-255) to a printable Unicode character.
    Printable ASCII bytes map to themselves.
    Non-printable bytes get mapped to characters starting at U+0100.
    (将每个字节 0-255 映射为可打印的 Unicode 字符:可打印 ASCII 字节映射到自身,不可打印字节映射到 U+0100 之后的字符。)
    """
    # Bytes that are already printable ASCII(本身已是可打印 ASCII 的字节)
    bs = list(range(ord("!"), ord("~") + 1))  # 33-126
    bs += list(range(ord("\xa1"), ord("\xac") + 1))  # 161-172
    bs += list(range(ord("\xae"), ord("\xff") + 1))  # 174-255
    
    cs = bs[:]
    # Non-printable bytes get mapped to U+0100 onwards(不可打印字节映射到 U+0100 之后)
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    
    return dict(zip(bs, [chr(c) for c in cs]))

byte_encoder = bytes_to_unicode()
byte_decoder = {v: k for k, v in byte_encoder.items()}

# Show the key mapping(展示几个关键映射)
print("Byte encoding examples:")
print(f"  Space (0x20) -> '{byte_encoder[32]}'  (this is the Ġ you see everywhere)")
print(f"  'a' (0x61)   -> '{byte_encoder[97]}'   (printable ASCII maps to itself)")
print(f"  'A' (0x41)   -> '{byte_encoder[65]}'   (same)")
print(f"  Tab (0x09)   -> '{byte_encoder[9]}'")
print(f"  Newline(0x0a)-> '{byte_encoder[10]}'")

Byte encoding examples:
  Space (0x20) -> 'Ġ'  (this is the Ġ you see everywhere)
  'a' (0x61)   -> 'a'   (printable ASCII maps to itself)
  'A' (0x41)   -> 'A'   (same)
  Tab (0x09)   -> 'ĉ'
  Newline(0x0a)-> 'Ċ'


## Step 2: How BPE Encoding Works(第二步:BPE 编码是怎么工作的)

BPE encoding converts text to token IDs in three steps:

> BPE 编码分三步把文本转成词元 ID:

```
  "The cat"  
       │
       ▼  Step A: Convert each byte to GPT-2's unicode char
  ["T", "h", "e", "Ġ", "c", "a", "t"]
       │
       ▼  Step B: Repeatedly merge the highest-priority adjacent pair
  ["The", "Ġcat"]         (after many merges)
       │
       ▼  Step C: Look up each merged token in vocab.json
  [464, 9246]
```

> (图示:步骤 A,把每个字节转成 GPT-2 的 unicode 字符;步骤 B,反复合并优先级最高的相邻对,多轮合并后得到 ["The", "Ġcat"];步骤 C,在 vocab.json 里查表得到 [464, 9246])

The merge rules in `merges.txt` are ordered by priority (most common pairs first).  
This is like a compression algorithm — the most frequent byte pairs get merged first.

> `merges.txt` 里的合并规则按优先级排序(最常见的字节对在前)。  
> 这类似压缩算法——最高频的字节对最先被合并。

In [5]:
# Look at the first and last merge rules(看看最前和最后的几条合并规则)
print("First 10 merge rules (highest priority = most common pairs):")
for i, merge in enumerate(merges_raw[:10]):
    pair = tuple(merge.split())
    print(f"  {i:>5d}: '{pair[0]}' + '{pair[1]}' -> '{pair[0]}{pair[1]}'")

print(f"\nLast 5 merge rules (lowest priority):")
for i, merge in enumerate(merges_raw[-5:], len(merges_raw)-5):
    pair = tuple(merge.split())
    print(f"  {i:>5d}: '{pair[0]}' + '{pair[1]}' -> '{pair[0]}{pair[1]}'")

print(f"\nTotal merge rules: {len(merges_raw):,}")
print(f"\nThe first merge 'Ġ' + 't' means: the space-t combination")
print(f"is the most common adjacent pair in GPT-2's training data.")

First 10 merge rules (highest priority = most common pairs):
      0: 'Ġ' + 't' -> 'Ġt'
      1: 'Ġ' + 'a' -> 'Ġa'
      2: 'h' + 'e' -> 'he'
      3: 'i' + 'n' -> 'in'
      4: 'r' + 'e' -> 're'
      5: 'o' + 'n' -> 'on'
      6: 'Ġt' + 'he' -> 'Ġthe'
      7: 'e' + 'r' -> 'er'
      8: 'Ġ' + 's' -> 'Ġs'
      9: 'a' + 't' -> 'at'

Last 5 merge rules (lowest priority):
  49995: 'om' + 'inated' -> 'ominated'
  49996: 'Ġreg' + 'ress' -> 'Ġregress'
  49997: 'ĠColl' + 'ider' -> 'ĠCollider'
  49998: 'Ġinform' + 'ants' -> 'Ġinformants'
  49999: 'Ġg' + 'azed' -> 'Ġgazed'

Total merge rules: 50,000

The first merge 'Ġ' + 't' means: the space-t combination
is the most common adjacent pair in GPT-2's training data.


## Step 3: Build the Encoder(第三步:实现编码器)

Now we implement BPE encoding. The algorithm:

> 现在我们来实现 BPE 编码。算法如下:

1. Split the input text into individual unicode-encoded bytes
2. Find all adjacent pairs
3. Find the pair with the **lowest merge index** (highest priority)
4. Merge that pair everywhere it appears
5. Repeat from step 2 until no more merges apply
6. Look up each final token in the vocabulary

> 1. 把输入文本拆成单个的 unicode 编码字节
> 2. 找出所有相邻对
> 3. 找到**合并序号最低**(优先级最高)的那一对
> 4. 把该对在序列中出现的每一处都合并
> 5. 回到第 2 步重复,直到没有可合并的为止
> 6. 把最终的每个词元拿到词表里查 ID

It's like a greedy compression pass — always merge the most common pattern first.

> 它就像一趟贪心压缩——永远先合并最常见的模式。

In [6]:
import re

# Build merge ranking: (pair) -> priority (lower = merge first)(构建合并优先级表:(字节对) -> 序号,序号越小越先合并)
bpe_ranks = {}
for i, merge in enumerate(merges_raw):
    pair = tuple(merge.split())
    bpe_ranks[pair] = i

# GPT-2's regex pattern for splitting text into preliminary chunks(GPT-2 用于把文本预切分成块的正则)
# This splits on word boundaries, keeping spaces attached to the following word(按词边界切分,空格跟随在后面的词上)
GPT2_PATTERN = re.compile(
    r"""'s|'t|'re|'ve|'m|'ll|'d| ?\w+| ?[^\s\w]+|\s+(?!\S)|\s+"""
)

def get_pairs(tokens):
    """Get all adjacent pairs from a list of tokens.(取词元列表中的所有相邻对。)"""
    return set(zip(tokens[:-1], tokens[1:]))

def bpe_merge(tokens):
    """Repeatedly merge the highest-priority pair until no merges remain.(反复合并优先级最高的相邻对,直到没有可合并的为止。)"""
    while True:
        pairs = get_pairs(tokens)
        if not pairs:
            break
        
        # Find the pair with the lowest rank (highest priority)(找出序号最小的那一对(即优先级最高))
        best_pair = min(pairs, key=lambda p: bpe_ranks.get(p, float("inf")))
        
        if best_pair not in bpe_ranks:
            break  # No more known merges(没有已知的合并规则了)
        
        # Merge that pair everywhere in the token list(把该对在整个词元列表中出现的每一处都合并)
        first, second = best_pair
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == first and tokens[i+1] == second:
                new_tokens.append(first + second)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    
    return tokens

def encode(text):
    """Encode text to token IDs using GPT-2 BPE.(用 GPT-2 的 BPE 把文本编码成词元 ID。)"""
    token_ids = []
    
    # Split text into preliminary chunks(把文本切成初始块)
    chunks = re.findall(GPT2_PATTERN, text)
    
    for chunk in chunks:
        # Convert each byte to GPT-2's unicode representation(把每个字节转成 GPT-2 的 unicode 表示)
        unicode_chars = [byte_encoder[b] for b in chunk.encode("utf-8")]
        
        # Apply BPE merges(执行 BPE 合并)
        merged = bpe_merge(unicode_chars)
        
        # Look up each merged token in the vocabulary(把合并后的每个词元拿到词表里查 ID)
        for token in merged:
            token_ids.append(vocab[token])
    
    return token_ids

print("Encoder built. Let's test it.")

Encoder built. Let's test it.


## Step 4: Test the Encoder(第四步:测试编码器)

Let's encode some text and trace the process step by step.

> 来编码一些文本,逐步跟踪整个过程。

In [7]:
# Encode and show results(编码并展示结果)
test_cases = [
    "The cat sat",
    "Inference engineering makes AI apps fast.",
    "Kubernetes",
    "NVIDIA",
    "192.168.1.76",
    "unhappiness",
]

for text in test_cases:
    ids = encode(text)
    tokens = [id_to_token[i] for i in ids]
    print(f"Text:   {repr(text)}")
    print(f"IDs:    {ids}")
    print(f"Tokens: {tokens}")
    print(f"Count:  {len(ids)} tokens for {len(text)} chars")
    print()

Text:   'The cat sat'
IDs:    [464, 3797, 3332]
Tokens: ['The', 'Ġcat', 'Ġsat']
Count:  3 tokens for 11 chars

Text:   'Inference engineering makes AI apps fast.'
IDs:    [818, 4288, 8705, 1838, 9552, 6725, 3049, 13]
Tokens: ['In', 'ference', 'Ġengineering', 'Ġmakes', 'ĠAI', 'Ġapps', 'Ġfast', '.']
Count:  8 tokens for 41 chars

Text:   'Kubernetes'
IDs:    [42, 18478, 3262, 274]
Tokens: ['K', 'uber', 'net', 'es']
Count:  4 tokens for 10 chars

Text:   'NVIDIA'
IDs:    [38021]
Tokens: ['NVIDIA']
Count:  1 tokens for 6 chars

Text:   '192.168.1.76'
IDs:    [17477, 13, 14656, 13, 16, 13, 4304]
Tokens: ['192', '.', '168', '.', '1', '.', '76']
Count:  7 tokens for 12 chars

Text:   'unhappiness'
IDs:    [403, 71, 42661]
Tokens: ['un', 'h', 'appiness']
Count:  3 tokens for 11 chars



## Step 5: Watch BPE Merges Happen(第五步:亲眼看看 BPE 合并过程)

Let's trace the merge process step by step for one word to see exactly how BPE works.

> 我们对一个词逐步跟踪合并过程,看看 BPE 到底是怎么工作的。

In [8]:
def bpe_merge_verbose(tokens):
    """Same as bpe_merge, but prints each step.(与 bpe_merge 相同,但会打印每一步。)"""
    step = 0
    print(f"  Start: {tokens}")
    
    while True:
        pairs = get_pairs(tokens)
        if not pairs:
            break
        
        best_pair = min(pairs, key=lambda p: bpe_ranks.get(p, float("inf")))
        if best_pair not in bpe_ranks:
            break
        
        rank = bpe_ranks[best_pair]
        first, second = best_pair
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == first and tokens[i+1] == second:
                new_tokens.append(first + second)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
        step += 1
        print(f"  Merge {step}: '{first}' + '{second}' (rank {rank}) -> {tokens}")
    
    return tokens

# Trace "Kubernetes"(跟踪 "Kubernetes" 的分词过程)
word = "Kubernetes"
print(f"Tracing BPE for: {repr(word)}")
print()
unicode_chars = [byte_encoder[b] for b in word.encode("utf-8")]
result = bpe_merge_verbose(unicode_chars)
print(f"\nFinal tokens: {result}")
print(f"Token IDs: {[vocab[t] for t in result]}")

Tracing BPE for: 'Kubernetes'

  Start: ['K', 'u', 'b', 'e', 'r', 'n', 'e', 't', 'e', 's']
  Merge 1: 'e' + 'r' (rank 7) -> ['K', 'u', 'b', 'er', 'n', 'e', 't', 'e', 's']
  Merge 2: 'e' + 's' (rank 18) -> ['K', 'u', 'b', 'er', 'n', 'e', 't', 'es']
  Merge 3: 'e' + 't' (rank 60) -> ['K', 'u', 'b', 'er', 'n', 'et', 'es']
  Merge 4: 'b' + 'er' (rank 271) -> ['K', 'u', 'ber', 'n', 'et', 'es']
  Merge 5: 'n' + 'et' (rank 3006) -> ['K', 'u', 'ber', 'net', 'es']
  Merge 6: 'u' + 'ber' (rank 18222) -> ['K', 'uber', 'net', 'es']

Final tokens: ['K', 'uber', 'net', 'es']
Token IDs: [42, 18478, 3262, 274]


## Step 6: Build the Decoder(第六步:实现解码器)

Decoding is simpler — just reverse the lookup and convert bytes back to text.

> 解码更简单——反着查表,再把字节转回文本就行。

In [9]:
def decode(token_ids):
    """Decode token IDs back to text.(把词元 ID 解码回文本。)"""
    # Look up each ID -> token string(逐个查 ID -> 词元字符串)
    token_strings = [id_to_token[i] for i in token_ids]
    
    # Concatenate all token strings(把所有词元字符串拼接起来)
    combined = "".join(token_strings)
    
    # Convert GPT-2 unicode chars back to bytes, then to text(把 GPT-2 的 unicode 字符转回字节,再解码成文本)
    byte_values = [byte_decoder[c] for c in combined]
    return bytes(byte_values).decode("utf-8", errors="replace")

# Verify round-trip: encode then decode should give back the original(验证往返:先编码再解码,应还原出原文)
for text in test_cases:
    ids = encode(text)
    decoded = decode(ids)
    match = "OK" if decoded == text else "MISMATCH"
    print(f"  [{match}] {repr(text)} -> {ids} -> {repr(decoded)}")

  [OK] 'The cat sat' -> [464, 3797, 3332] -> 'The cat sat'
  [OK] 'Inference engineering makes AI apps fast.' -> [818, 4288, 8705, 1838, 9552, 6725, 3049, 13] -> 'Inference engineering makes AI apps fast.'
  [OK] 'Kubernetes' -> [42, 18478, 3262, 274] -> 'Kubernetes'
  [OK] 'NVIDIA' -> [38021] -> 'NVIDIA'
  [OK] '192.168.1.76' -> [17477, 13, 14656, 13, 16, 13, 4304] -> '192.168.1.76'
  [OK] 'unhappiness' -> [403, 71, 42661] -> 'unhappiness'


## Why Tokenization Matters for Inference(分词为什么对推理很重要)

Tokenization affects inference performance directly:

> 分词直接影响推理性能:

| Factor | Impact |
|--------|--------|
| **Tokens per request** | More tokens = more forward passes = slower response |
| **Vocabulary size** | Larger vocab = larger embedding table = more GPU memory |
| **Token efficiency** | Better tokenizer = fewer tokens for same text = faster + cheaper |

> (表中:每请求词元数——词元越多 = 前向传播次数越多 = 响应越慢;词表大小——词表越大 = 嵌入表越大 = GPU 显存占用越多;词元效率——分词器越好 = 同样文本词元越少 = 又快又省钱)

This is why newer models (Llama 3, Qwen) use larger vocabularies (128K+) with more efficient tokenization. Fewer tokens per request = faster inference.

> 这就是为什么更新的模型(Llama 3、Qwen)采用更大的词表(128K+)和更高效的分词。每个请求的词元越少 = 推理越快。

In [10]:
# Token efficiency: how many characters per token?(词元效率:平均每个词元对应多少个字符?)
sample_text = (
    "The inference engine processes requests by loading model weights into GPU memory, "
    "tokenizing the input, running the forward pass through transformer blocks, and "
    "sampling output tokens autoregressively until the stop condition is met."
)

ids = encode(sample_text)
tokens = [id_to_token[i] for i in ids]

print(f"Text length:   {len(sample_text)} characters")
print(f"Token count:   {len(ids)} tokens")
print(f"Chars/token:   {len(sample_text) / len(ids):.1f}")
print(f"\nTokenized:")

# Show tokens with boundaries(展示词元及其边界)
line = ""
for t in tokens:
    decoded_t = bytes([byte_decoder[c] for c in t]).decode("utf-8", errors="replace")
    line += f"[{decoded_t}]"
print(f"  {line}")

Text length:   233 characters
Token count:   41 tokens
Chars/token:   5.7

Tokenized:
  [The][ inference][ engine][ processes][ requests][ by][ loading][ model][ weights][ into][ GPU][ memory][,][ token][izing][ the][ input][,][ running][ the][ forward][ pass][ through][ transformer][ blocks][,][ and][ sampling][ output][ tokens][ aut][ore][gress][ively][ until][ the][ stop][ condition][ is][ met][.]


## Key Takeaways(核心要点)

- Tokenization is **not** a neural network — it's a deterministic lookup + merge algorithm.
- GPT-2 uses **Byte Pair Encoding**: start with individual bytes, greedily merge the most common pairs.
- Common words like "the" become 1 token. Rare words like "Kubernetes" get split into subwords.
- The byte-level encoding (`Ġ` = space) means the tokenizer can handle **any input** without unknown tokens.
- Token count directly impacts inference cost — fewer tokens = fewer forward passes = faster.

> - 分词**不是**神经网络——它是确定性的"查表 + 合并"算法。
> - GPT-2 使用**字节对编码(BPE)**:从单个字节出发,贪心地合并最常见的字节对。
> - "the" 这类常见词只占 1 个词元;"Kubernetes" 这类生僻词会被拆成子词。
> - 字节级编码(`Ġ` = 空格)意味着分词器可以处理**任何输入**,不会出现未知词元。
> - 词元数量直接影响推理成本——词元越少 = 前向传播越少 = 越快。

## Next(下一步)

**Notebook 03** — We use these token IDs to look up **embeddings** from `wte.weight` — converting integers into the 768-dimensional vectors that the model actually works with.

> **Notebook 03**——我们用这些词元 ID 去 `wte.weight` 里查**嵌入(embeddings)**——把整数变成模型真正处理的 768 维向量。

## References(参考资料)

- *Inference Engineering* Ch 2.2 (pp. 46–47) — Tokenization, Subword Tokenization
- Sennrich et al., "Neural Machine Translation of Rare Words with Subword Units" (2016) — BPE for NLP

> - 《Inference Engineering》第 2.2 章(第 46–47 页)——分词、子词分词
> - Sennrich 等,《Neural Machine Translation of Rare Words with Subword Units》(2016)——把 BPE 引入 NLP 的经典论文